# CI/CD in the Cloud {background-color="black" background-image="https://images.unsplash.com/photo-1451187580459-43490279c0fa?w=1600" background-size="cover" background-opacity="0.5"}

> *GitOps on AWS with GitHub Actions and OIDC*

## The story so far

::::{.columns}

::: {.fragment .column width="50%"}

**GitOps locally:**

| Step | Tool | Result |
|------|------|--------|
| Git host | Gitea (self-hosted) | Repo + CI/CD on your laptop |
| Runner | `act_runner` | Runs jobs on localhost |
| Infrastructure | Multipass VMs | Local virtualisation |
| Credentials | Gitea Secrets (static keys) | Stored in Gitea |
| State | `terraform.tfstate` on disk | Single machine only |

:::

::: {.fragment .column width="50%"}

**GitOps in the cloud:**

| Step | Tool | Result |
|------|------|--------|
| Git host | **GitHub** | SaaS, no server to run |
| Runner | **GitHub-hosted** | Ephemeral `ubuntu-latest` |
| Infrastructure | **AWS EC2** | Real cloud instances |
| Credentials | **OIDC** | No stored AWS keys |
| State | **S3 + DynamoDB** | Shared, locked, versioned |


:::
::::


# GitHub Actions {background-color="#161b22" background-image="https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png" background-size="10%" background-position="88% 50%"}

## GitHub Actions vs Gitea Actions

::::{.columns}

::: {.fragment .column width="50%"}

:::{.fragment .grid}
**Same**

- YAML syntax: `on:`, `jobs:`, `steps:`, `uses:`, `run:`
- marketplace actions: `actions/checkout@v4`, `opentofu/setup-opentofu@v1`
- secrets model: `${{ secrets.MY_SECRET }}`
- `needs:` for job dependencies

:::

:::{.fragment}
**changes:**

- Workflow path: `.gitea/workflows/` → `.github/workflows/`
- Context object: `gitea.sha` → `github.sha`
- Runner: `self-hosted` → `ubuntu-latest` (hosted)
- Runner setup: Register `act_runner` → Nothing — it's managed
:::

:::

::: {.fragment .column width="50%"}

**GitHub-hosted runners:**

- Fresh VM per job — no state left between runs
- Pre-installed: `git`, `docker`, `aws-cli`, `python`, `node`, …
- Free tier: 2 000 min/month (public repos: unlimited)
- `terraform.tfstate` cannot live on the runner disk, a remote backend is required
:::

::::

# Prerequisites {background-color="#1A1A2E"}

## Free AWS account (not AWS Academy)

::::{.columns}

::: {.fragment .column width="50%"}

**AWS Academy limitations that matter here:**

| Feature | AWS Academy | Free Account |
|---------|-------------|--------------|
| OIDC Identity Provider | ❌ `iam:CreateOpenIDConnectProvider` blocked | ✅ |
| Custom IAM Roles | ❌ Trust policy editing blocked | ✅ |
| Session duration | ⚠️ 4 hours, then credentials expire | ✅ Unlimited |
| Account ownership | Shared lab environment | ✅ Yours |
| After course ends | Account deleted | ✅ Keep it |

:::

::: {.fragment .column width="50%"}

**Free tier covers everything in this lab:**

| Service | Free tier |
|---------|-----------|
| EC2 `t2.micro` | 750 h/month for 12 months |
| S3 | 5 GB storage, 20 000 GET, 2 000 PUT |
| DynamoDB | 25 GB, 25 WCU/RCU |
| Data transfer | 100 GB/month outbound |


::: {.callout-note}
Enable **MFA** on the root account and create an **IAM user** for daily use — never use root credentials in workflows.
:::

:::
::::

## Configure AWS CLI

::::{.columns}

::: {.fragment .column width="50%"}

**Configure AWS CLI credentials:**

```bash
aws configure --profile nicaws
```

```
AWS Access Key ID:     <from IAM user>
AWS Secret Access Key: <from IAM user>
Default region name:   eu-south-1
Default output format: json
```

**Verify:**

```bash
aws sts get-caller-identity --profile nicaws
```

:::

::: {.fragment .column width="50%"}

**Where to get the credentials:**

```
AWS Console → IAM → Users → <your user>
  → Security credentials
    → Create access key
      → CLI use case
```

::: {.callout-warning}
Never use **root account** credentials. Create a dedicated IAM user with `AdministratorAccess` for the lab, then delete the access key when done.
:::

::: {.callout-important}
`aws sts get-caller-identity` must succeed before running any `tofu` or `aws` CLI command — if credentials are invalid or expired, everything downstream will fail.
:::

:::
::::

## Fork the template repo

::::{.columns}

::: {.fragment .column width="50%"}

**Fork on GitHub:**

<https://github.com/unict-cloud-systems/github-actions-aws>

Fork → Create fork (under your username)

**Clone your fork:**

```bash
git clone git@github.com:YOUR-USERNAME/github-actions-aws.git
cd github-actions-aws
```

:::

::: {.fragment .column width="50%"}

::: {.callout-tip}
Keep the fork URL (`YOUR-USERNAME/github-actions-aws`) handy — you'll need it when configuring the IAM trust policy.
:::

:::
::::

## Setting up the S3 backend

::::{.columns}

::: {.fragment .column width="50%"}

**Set environment variables first:**

```bash
export AWS_PROFILE="nicaws"                               # if using a named profile
export TF_STATE_BUCKET="YOUR-NAME-unict-cloud-systems-tofu-state"
export AWS_DEFAULT_REGION="eu-south-1"   # Milan
```

**One-time setup via AWS CLI:**

```bash
# Create the bucket (name must be globally unique)
aws s3api create-bucket \
  --bucket "$TF_STATE_BUCKET" \
  --region "$AWS_DEFAULT_REGION" \
  --create-bucket-configuration LocationConstraint="$AWS_DEFAULT_REGION"

# Enable versioning
aws s3api put-bucket-versioning \
  --bucket "$TF_STATE_BUCKET" \
  --versioning-configuration Status=Enabled

# Create DynamoDB table for state locking
aws dynamodb create-table \
  --table-name tofu-state-lock \
  --attribute-definitions AttributeName=LockID,AttributeType=S \
  --key-schema AttributeName=LockID,KeyType=HASH \
  --billing-mode PAY_PER_REQUEST
```

:::

::: {.fragment .column width="50%"}

**Update `terraform/providers.tf` in your fork:**

```hcl
backend "s3" {
  bucket         = "YOUR-NAME-tofu-state"   # ← change this
  key            = "lab/ec2/terraform.tfstate"
  region         = "eu-south-1"
  dynamodb_table = "tofu-state-lock"
  encrypt        = true
}
```

**Initialise the backend locally:**

```bash
export AWS_PROFILE="nicaws"  # if using a named profile
cd terraform && tofu init
```

::: {.callout-note}
`tofu init` migrates any existing local state to S3 on first run.  
`--create-bucket-configuration` is required for all regions except `us-east-1`.  
OpenTofu reads `AWS_PROFILE` — it does **not** support `--profile` flags.
:::

:::
::::

## IAM Role for GitHub Actions (OIDC)

::::{.columns}

::: {.fragment .column width="50%"}

**Step 1 — Add GitHub as an OIDC Identity Provider in IAM:**

```
IAM Console
  → Identity Providers → Add provider
    → Provider type: OpenID Connect
    → Provider URL: https://token.actions.githubusercontent.com
    → Audience (Destinatari): sts.amazonaws.com
    → Add provider
```

**Step 2 — Create an IAM Role:**

```
IAM Console → Roles → Create role
  → Trusted entity: Web identity
  → Identity provider: token.actions.githubusercontent.com
  → Audience: sts.amazonaws.com
  → GitHub organization: YOUR-USERNAME   ← your GitHub username
  → GitHub repository: github-actions-aws  ← optional
  → GitHub branch: main                    ← optional
  → Permissions: AmazonEC2FullAccess,
                 AmazonS3FullAccess,
                 AmazonDynamoDBFullAccess
```

::: {.callout-note}
**Audience = Destinatari** nella console AWS in italiano.  
**GitHub organization** accetta anche uno username personale — non serve un'organizzazione.  
Repository e branch sono opzionali ma riducono la superficie di attacco.  
**DynamoDB** è necessario per il state locking del backend S3.
:::

:::

::: {.fragment .column width="50%"}

**The resulting trust policy (for reference):**

```json
{
  "Version": "2012-10-17",
  "Statement": [{
    "Effect": "Allow",
    "Principal": {
      "Federated": "arn:aws:iam::ACCOUNT:oidc-provider/token.actions.githubusercontent.com"
    },
    "Action": "sts:AssumeRoleWithWebIdentity",
    "Condition": {
      "StringEquals": {
        "token.actions.githubusercontent.com:aud": "sts.amazonaws.com"
      },
      "StringLike": {
        "token.actions.githubusercontent.com:sub":
          "repo:YOUR-USERNAME/github-actions-aws:ref:refs/heads/main"
      }
    }
  }]
}
```

::: {.callout-tip}
Use `repo:YOUR-USERNAME/github-actions-aws:*` (wildcard) during setup to allow all branches — tighten to `ref:refs/heads/main` afterwards.
:::

:::
::::

## Configure GitHub Actions secrets

::::{.columns}

::: {.fragment .column width="50%"}

**Generate SSH key pair** (needed by Ansible to SSH into EC2):

```bash
ssh-keygen -t ed25519 -f id_ed25519 -N ""
# id_ed25519     → SSH_PRIVATE_KEY secret
# id_ed25519.pub → SSH_PUBLIC_KEY  secret
```

::: {.callout-warning}
`id_ed25519` and `id_ed25519.pub` are in `.gitignore` — never commit them.
:::

**Add secrets to your fork:**

```
Settings → Secrets and variables → Actions → New secret

AWS_ROLE_ARN    = arn:aws:iam::ACCOUNT:role/github-actions-role
SSH_PRIVATE_KEY = <contents of id_ed25519>
SSH_PUBLIC_KEY  = <contents of id_ed25519.pub>
```

:::

::: {.fragment .column width="50%"}

**What each secret is used for:**

| Secret | Used by |
|--------|---------|
| `AWS_ROLE_ARN` | `infra.yml` — OIDC role assumption |
| `SSH_PRIVATE_KEY` | `configure.yml` — Ansible SSH into EC2 |
| `SSH_PUBLIC_KEY` | `infra.yml` — injected into EC2 at provision time |

::: {.callout-tip}
`SSH_PUBLIC_KEY` goes into the EC2 instance via Terraform so that the `configure` workflow can connect to it. Both keys must be generated together as a pair.
:::

:::
::::

## Branch protection rules

::::{.columns}

::: {.fragment .column width="50%"}

**Protect `main` in GitHub:**

```
Settings → Branches → Add rule
  Branch name pattern: main

  ✅ Require a pull request before merging
      Required approvals: 1

  ✅ Require status checks to pass before merging
      Status checks: plan (infra.yml)

  ✅ Do not allow bypassing the above settings
```

:::

::: {.fragment .column width="50%"}

**Result:**

- Direct push to `main` → blocked
- PR without a passing `plan` job → cannot merge
- PR without 1 approval → cannot merge

**GitHub Environments for `apply`:**

```
Settings → Environments → New: production
  ✅ Required reviewers: <your name>
  ✅ Deployment branches: main only
```

The `apply` job pauses and waits for a human to click *Approve and deploy* in the Actions UI before running `tofu apply`.

::: {.callout-note}
**Lab:** non puoi approvare la tua stessa PR.  
Lavorate in coppia — uno approva la PR dell'altro.  
In alternativa, disabilita temporaneamente *"Do not allow bypassing"* nelle impostazioni del branch.
:::

:::
::::

# The GitOps Pipeline {background-color="#1A1A2E"}

## AWS Credentials

Accessing AWS from a pipeline requires credentials. There are two main approaches:
- AWS keys (secret access key + access key ID)
- OIDC (OpenID Connect) federated identity

::::{.columns}

::: {.fragment .column width="50%"}

Static keys when used locally it's possible to use environment variables or store in Gitea Secrets. 
In GitHub Actions, we can do the same with GitHub Secrets and add them to the job environment:

```yaml
  env:
    AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
    AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
```


:::

::: {.fragment .column width="50%"}

However, long-lived IAM access keys stored as GitHub Secrets have several risks:

- Keys never expire — if leaked, attacker has indefinite access
- Hard to rotate across multiple repos / pipelines
- Keys have the same permissions whether used by a human or a pipeline
- Violates least-privilege at the *time* dimension

::: {.callout-warning .fragment}
GitHub Secret scanning detects accidentally committed keys — but the pipeline itself is still a risk surface.
:::

:::
::::

## OpenID Connect

Github Actions supports OIDC federated identity with AWS STS. This allows the runner to request temporary credentials from AWS without any long-lived keys stored in GitHub.

::::{.columns}

::: {.fragment .column width="55%"}

**How it works:**

```{mermaid}
sequenceDiagram
    participant GH as GitHub Actions
    participant STS as AWS STS
    participant IAM as IAM Role
    GH->>STS: AssumeRoleWithWebIdentity (OIDC token)
    STS->>IAM: Validate trust policy
    IAM-->>STS: Allowed
    STS-->>GH: Temporary credentials (1h TTL)
    GH->>GH: Run tofu / aws cli with temp creds
```

:::

::: {.fragment .column width="45%"}

**What you store in GitHub Secrets:**

```
AWS_ROLE_ARN = arn:aws:iam::123456789:role/github-actions-role
```

That's it. No `AWS_ACCESS_KEY_ID`. No `AWS_SECRET_ACCESS_KEY`.

**Token properties:**

- Expires after ~1 hour
- Scoped to the exact repo + branch in the trust policy
- Cannot be reused outside the job that requested it

::: {.callout-tip .fragment}
This is the **production standard** for GitHub Actions + AWS. If a key leaks, it's already expired.
:::

:::
::::

## Why a remote backend?

::::{.columns}

::: {.fragment .column width="50%"}

**Problem with local state on CI runners:**

```
Job 1  (runner A): tofu apply → writes terraform.tfstate
                                ↓ runner destroyed
Job 2  (runner B): tofu plan  → no state file! 💥
```

- GitHub-hosted runners are ephemeral — workspace is deleted after every job
- Without shared state, `tofu plan` can't know what already exists
- Two concurrent runs could corrupt state (race condition)

:::

::: {.fragment .column width="50%"}

**S3 backend solves all three problems:**

| Problem | Solution |
|---------|----------|
| State not persisted | S3 bucket (durable, versioned) |
| Concurrent runs | DynamoDB lock table |
| State contains secrets | S3 bucket private + SSE |

::: {.callout-tip}
Enable **S3 versioning** — every `tofu apply` creates a new version.  
Rollback = restore previous version.
:::

:::
::::

## Pipeline design

::::{.columns}

::: {.fragment .column width="50%"}

```{mermaid}
flowchart TD
    PR["Open PR"] --> PLAN["plan job"]
    PLAN --> REVIEW["Human review + approve"]
    REVIEW --> MERGE["Merge to main"]
    MERGE --> APPLY["apply job"]
    APPLY --> CONFIGURE["configure workflow"]
    CONFIGURE --> DONE["EC2 ready ✅"]
```

:::

::: {.fragment .column width="50%"}

**Two workflows:**

| File | Trigger | Jobs |
|------|---------|------|
| `infra.yml` | push/PR on `terraform/**` | `plan` (PR), `apply` (main) |
| `configure.yml` | `workflow_run` after infra | install Docker via Ansible |

**Key rules:**
- `plan` runs on **every PR** — no infra change without a reviewed plan
- `apply` only runs on **merge to main**
- `configure` only runs after a **successful** `apply`

:::
::::

## `.github/workflows/infra.yml`

Define the infrastructure pipeline using stadard OpenTofu cycles:

- `plan`: Post the plan output as a comment on the PR for review.
-   `apply`: Only runs on `main` branch, applies the changes, and triggers the configuration workflow.

:::{.callout-note .fragment}
See file in the repo for the full code with comments.
:::

Let's test the pipeline by first creating a branch and making a change to the infrastructure code — for example, changing the EC2 number of instances from 1 to 2 in `terraform/ec2/variables.tf`.

- Then create a new PR with a plan comment, and after approval
- execution of the apply workflow. 

Check the Actions tab for logs and the AWS Console to see the changes.

## `.github/workflows/configure.yml`

::::{.columns}

::: {.fragment .column width="55%"}

```yaml
name: Configure EC2

on:
  workflow_run:
    workflows: ["Provision AWS EC2"]
    types: [completed]
    branches: [main]

jobs:
  configure:
    runs-on: ubuntu-latest
    if: ${{ github.event.workflow_run.conclusion == 'success' }}

    steps:
      - uses: actions/checkout@v4

      - name: Download EC2 IP
        uses: actions/download-artifact@v4
        with:
          name: ec2-ip
          github-token: ${{ secrets.GITHUB_TOKEN }}
          run-id: ${{ github.event.workflow_run.id }}

      - name: Install Ansible
        run: pip install ansible

      - name: Write SSH private key
        run: |
          echo "${{ secrets.SSH_PRIVATE_KEY }}" > id_ed25519
          chmod 600 id_ed25519

      - name: Install Docker via Ansible
        run: |
          EC2_IP=$(cat ec2_ip.txt)
          ansible-playbook \
            -i "${EC2_IP}," \
            --private-key id_ed25519 \
            -u ubuntu \
            ansible/install_docker.yml
```

:::

::: {.fragment .column width="45%"}

**Key points:**

- Triggered by `workflow_run` — not `push` — so it only runs after a successful infra apply
- EC2 IP passed between workflows via **Actions Artifacts** (ephemeral, 1 day retention)
- SSH private key comes from GitHub Secret — written to disk only for the duration of the job
- Ansible inventory built inline: `"${EC2_IP},"` — the trailing comma makes it a valid single-host inventory

:::
::::

# The GitOps Loop {background-color="#1A1A2E"}

## Step 1 — Confirm your fork is ready

::::{.columns}

::: {.fragment .column width="50%"}

**You should already have:**

- ✅ Repo forked at `github.com/YOUR-USERNAME/lab-gitops-aws`
- ✅ SSH key pair generated (`id_ed25519` / `id_ed25519.pub`)
- ✅ `SSH_PRIVATE_KEY` and `SSH_PUBLIC_KEY` set in fork Secrets
- ✅ `AWS_ROLE_ARN` set in fork Secrets (from OIDC setup)
- ✅ Bucket name updated in `terraform/providers.tf`

:::

::: {.fragment .column width="50%"}

**Initialise the backend locally:**

```bash
cd terraform && tofu init
```

**Verify AWS credentials work:**

```bash
aws sts get-caller-identity
```

::: {.callout-tip}
If `tofu init` fails, check that the S3 bucket exists and the AWS CLI is configured with the correct region (`us-east-1`).
:::

:::
::::

## Step 2 — Open a PR, review the plan

::::{.columns}

::: {.fragment .column width="50%"}

**Make a change on a feature branch:**

```bash
git checkout -b feat/add-tag
```

Edit `terraform/main.tf` — add a tag to the EC2 instance:

```hcl
tags = {
  Name        = "unict-cloud-systems"
  Environment = "lab"
}
```

```bash
git add terraform/main.tf
git commit -m "chore: add environment tag to EC2"
git push origin feat/add-tag

# Open a PR on GitHub
```

::: {.callout-warning}
In `eu-south-1` (Milan) **only `t3.micro` is Free Tier eligible** — `t2.micro` is not available in this region.  
Make sure `variables.tf` has `default = "t3.micro"`.
:::

:::

::: {.fragment .column width="50%"}

**What happens:**

```
PR opened
  └─► plan job triggers
        ├─ aws-actions/configure-aws-credentials (OIDC)
        ├─ tofu init
        ├─ tofu plan
        └─ posts plan output as PR comment:

  ~ aws_instance.web  will be updated in-place
    + tags = {
        + "Environment" = "lab"
        + "Name"        = "unict-cloud-systems"
      }

  Plan: 0 to add, 1 to change, 0 to destroy.
```

Reviewer sees the exact infra diff before approving.

:::
::::

## Step 3 — Merge → apply → configure

::::{.columns}

::: {.fragment .column width="50%"}

**Merge the PR:**

```
GitHub PR → Merge pull request

Run #2  Provision AWS EC2
  ├─ plan   ✅
  └─ apply  ✅  (waiting for 'production' environment approval)
        → EC2 instance created
        → ec2_ip.txt artifact uploaded
```

:::

::: {.fragment .column width="50%"}

**configure workflow triggers automatically:**

```
Run #1  Configure EC2
  └─ configure  ✅
        ├─ Download EC2 IP artifact
        ├─ Install Ansible
        ├─ Write SSH key
        └─ ansible-playbook install_docker.yml
              TASK: Install docker.io  ok
              TASK: Start Docker       ok
```

**Verify:**

```bash
ssh -i terraform/id_ed25519 ubuntu@<EC2_IP>
docker info | grep 'Server Version'
```

:::
::::

## Step 4 — Rollback with `git revert`

::::{.columns}

::: {.fragment .column width="50%"}

**Rollback is just another commit:**

```bash
git revert HEAD
git push origin main
```

This triggers the pipeline again:

```
tofu plan:
  ~ aws_instance.web  will be replaced
    - instance_type = "t3.micro"
    + instance_type = "t2.micro"

tofu apply: instance replaced
```

:::

::: {.fragment .column width="50%"}

**Why this matters:**

- No manual intervention needed — Git history drives infrastructure
- Rollback has an audit trail (the revert commit)
- Same review process applies — even rollbacks go through PR + plan
- Compare to `tofu apply` from a laptop: no trail, no review, no visibility

::: {.callout-tip}
This is the **GitOps promise**: the repository is always the authoritative description of what is deployed.
:::

:::
::::

## Step 5 — Destroy workflow

::::{.columns}

::: {.fragment .column width="55%"}

**`.github/workflows/destroy.yml`:**

```yaml
name: Destroy AWS EC2

on:
  workflow_dispatch:    # ← manual trigger only

permissions:
  id-token: write
  contents: read

jobs:
  destroy:
    runs-on: ubuntu-latest
    environment: production
    steps:
      - uses: actions/checkout@v4
      - uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: ${{ secrets.AWS_ROLE_ARN }}
          aws-region: us-east-1
      - uses: opentofu/setup-opentofu@v1
      - run: tofu -chdir=terraform init
      - run: tofu -chdir=terraform destroy -auto-approve
```

:::

::: {.fragment .column width="45%"}

**Trigger manually:**

```
GitHub repo
  → Actions
    → Destroy AWS EC2
      → Run workflow
        → Branch: main
          → Run workflow ✅
```

::: {.callout-warning}
**Always run this at the end of the lab.**  
EC2 instances are billed per second — even stopped instances incur EBS charges.
:::

:::
::::

## Lab — GitOps on AWS

::::{.columns}

::: {.fragment .column width="50%"}

**Pre-lab checklist:**

- [ ] Free AWS account created at <https://aws.amazon.com/free>
- [ ] AWS CLI configured: `aws configure`
- [ ] S3 bucket + DynamoDB table created (one-time, see S3 backend slides)
- [ ] IAM OIDC provider added for GitHub
- [ ] IAM Role `github-actions-role` created with trust policy
- [ ] Template repo forked: `github.com/unict-cloud-systems/lab-gitops-aws`
- [ ] `AWS_ROLE_ARN` and `SSH_PRIVATE_KEY` set in **your fork's** Secrets

:::

::: {.fragment .column width="50%"}

**Lab steps:**

1. Fork the template repo on GitHub → clone your fork
2. Confirm S3 backend bucket name in `terraform/providers.tf`
3. Push to `main` → observe first `apply` run in Actions
4. Create a branch, change `instance_type` in `terraform/variables.tf`
5. Open a PR → read the `tofu plan` comment
6. Merge → watch `apply` + `configure` chain run
7. SSH into the EC2 instance and verify Docker is running
8. `git revert` → observe pipeline rolling back the instance type
9. Trigger the `Destroy AWS EC2` workflow manually

::: {.callout-warning}
Step 9 is **mandatory** — even free-tier EC2 has a 750 h/month limit shared across all instances.
:::

:::
::::